In [1]:
import chromadb
client = chromadb.Client()

In [3]:
# switch `create_collection` to `get_or_create_collection` to avoid creating a new collection every time
collection = client.get_or_create_collection(name="Nike_Ecommerce")
collection.count()

0

In [4]:
collection.query(
    query_texts=["Pineapple"],
    n_results=1
)

{'ids': [[]],
 'embeddings': None,
 'documents': [[]],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[]],
 'distances': [[]]}

In [8]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

loader = PyPDFLoader("testing.pdf")
pages = loader.load()  # list of Document objects

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(pages)


In [19]:
from pprint import pprint

# pprint(pages[0].metadata)
print("======================")
# pprint(pages[0].page_content)
# for chunk in chunks:
#     pprint(chunk.metadata)
#     pprint("******************")
#     pprint(chunk.page_content)
#     print("================================")

In [20]:
docs = [doc.page_content for doc in chunks]
ids = [f"chunk_{i}" for i in range(len(docs))]

In [24]:
# Use upsert to avoid duplicate insertions
collection.upsert(documents=docs, ids=ids)

In [52]:
query = "What is the warranty duration for Nike products?"
results = collection.query(
    query_texts=[query],
    n_results=1
)
pprint((results["documents"][0][0]))

('Product Warranty \n'
 'Nike stands firmly behind the quality of its products with a two-year '
 'warranty covering \n'
 'manufacturing defects in materials and workmanship. This warranty applies '
 'for up to two years from \n'
 'the date of manufacture (not the purchase date), which can be found on the '
 'product label inside the \n'
 'shoe or item. Defects such as faulty stitching, sole separation, or material '
 'flaws are covered.')


In [32]:
import nest_asyncio
nest_asyncio.apply()


In [ ]:
collection.count()

In [33]:
from agents import Agent, Runner, AsyncOpenAI, OpenAIChatCompletionsModel,RunConfig
import os
from dotenv import load_dotenv
load_dotenv()

external_client = AsyncOpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

model = OpenAIChatCompletionsModel(
    model="gemini-1.5-flash",
    openai_client=external_client
)

config = RunConfig(
    model=model,
    model_provider=external_client,
    tracing_disabled=True
)

In [43]:
from agents import function_tool

@function_tool
def rag_function(prompt:str)->str:
    """This function takes a string prompt as input and returns a string response."""
    results = collection.query(
    query_texts=[query],
    n_results=1
    )
    print("==Tool Used==")
    return results["documents"][0][0]

In [51]:
agent: Agent = Agent(name="Assistant", instructions="You are a Nike RAG agent so use your function and then give answer in formatted way",model=model,
                    tools=[rag_function]
                     )

result = Runner.run_sync(agent, "Tell me warrenty duration of nike products in years", run_config=config)

pprint(result)

==Tool Used==
RunResult(input='Tell me warrenty duration of nike products in years',
          new_items=[ToolCallItem(agent=Agent(name='Assistant',
                                              instructions='You are a Nike RAG '
                                                           'agent so use your '
                                                           'function and then '
                                                           'give answer in '
                                                           'formatted way',
                                              prompt=None,
                                              handoff_description=None,
                                              handoffs=[],
                                              model=<agents.models.openai_chatcompletions.OpenAIChatCompletionsModel object at 0x00000214DDBD9B50>,
                                              model_settings=ModelSettings(temperature=None,
                          